<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 11 — AI Cybersecurity

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 11 of 12 · 3 hours · Attacks the exact RR Finance system Module 10 deployed*

## Recap — what RR Finance already has, and why it is now a target

Ten modules built a real system: a baseline classifier (Module 1), an ANN and CNN with adversarial attacks (Module 2), NLP and a from-scratch language model (Modules 3–4), RAG and agents (Module 5), explainability and a fairness audit (Module 6), federated and differentially-private training (Module 7), multimodal pipelines (Module 8), graph-based fraud detection (Module 9), and finally — in Module 10 — a **real, running FastAPI service** that wraps Module 1's actual saved pipeline, validates input with Pydantic, and sits behind a container, Kubernetes manifests, and a mutual-TLS boundary.

Every previous module treated attacks as isolated demonstrations: poison *this* training set, fool *this* CNN, corrupt *this* federated update. Module 11 changes the frame. The moment Module 10 put the model behind an API a real loan-origination system could call, RR Finance acquired a real, reachable **attack surface**. This module does two things:

1. **Steps back and threat-models the whole system** — STRIDE against Module 10's architecture, and a MITRE ATLAS-style retrospective mapping every attack this course has already demonstrated onto a real, standard adversarial-ML taxonomy.
2. **Attacks the deployed model directly** — model extraction, evasion, and membership inference, each run against the *actual* Module 10 API, each measured honestly, and each followed by a real, tested defense.

Two of this module's findings are deliberately *un*-dramatic — an attack that fails, and a defense that only half-works. Those are kept in on purpose: knowing exactly *when* an attack does and does not work is the whole point of thinking like a defender.

In [ ]:
from pathlib import Path
import json, numpy as np

Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

print("Project folders ready: data/, artifacts/")
print("Module 11 attacks the deployed model -- it needs Module 1's real pipeline (loaded in Lesson 1).")

In [ ]:
%pip install -q fastapi uvicorn pydantic requests scikit-learn pandas joblib
print("Setup complete -- 'Requirement already satisfied' lines above are expected and fine.")
print("\nThis module stands up Module 10's REAL FastAPI service and attacks it over a real network")
print("connection where the environment allows one, falling back to an in-process call to the SAME")
print("real app object (same validation, same pipeline) where it does not -- disclosed in Lesson 4.")

---
## Lesson 1 — Recap: loading the exact system we are about to attack

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Load Module 1's real trained pipeline and confirm Module 10's API definition, so every attack below targets the real deployed system rather than a stand-in. |
| **2. Why does it matter in finance?** | An attack demonstrated against a toy model proves nothing about RR Finance's actual exposure. This module attacks the same object Modules 1, 6 and 10 all used. |
| **3. Why this technique?** | Load Module 1's real saved pipeline directly, exactly as every module since Module 2 has, and reconstruct Module 10's exact API contract around it. |
| **4. What do the parameters mean?** | N/A — this is a load step. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | If Module 1's artifact is missing, run Module 1's notebook first — there is nothing to attack without the real model. |

In [ ]:
import joblib

BASELINE_PATH = Path("artifacts/baseline_logreg_pipeline.joblib")
MODULE1_METRICS_PATH = Path("artifacts/module1_metrics.json")

if not BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {BASELINE_PATH.resolve()}.\n"
        "Run Module 1's notebook first -- this module attacks its REAL saved pipeline."
    )

pipeline = joblib.load(BASELINE_PATH)
print("Loaded Module 1's ACTUAL saved pipeline:", BASELINE_PATH.resolve())
print("Pipeline steps:", list(pipeline.named_steps.keys()))

if MODULE1_METRICS_PATH.exists():
    with open(MODULE1_METRICS_PATH) as f:
        module1_metrics = json.load(f)
    print(f"Victim model test AUC: {module1_metrics['baseline_logreg_test_auc']:.4f} "
          f"on {module1_metrics['dataset_rows']} applicants")

FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]
print("\nThis is the model behind Module 10's /predict endpoint. Everything below attacks THIS object.")

**Reading this honestly:** the victim is deliberately unglamorous — a `StandardScaler` feeding a `LogisticRegression`, with a test AUC around 0.69. That modesty matters for this module in a way it did not before. A simple, well-regularised linear model is *easy to steal* (Lesson 4) and *easy to fool* (Lesson 5), because its decision surface is smooth and low-complexity — but it is also *hard to extract training-set membership from* (Lesson 6), because it never had the capacity to memorise anyone. The same property is a weakness against one attack and a defence against another. That tension is the through-line of this module.

---
## Lesson 2 — Threat-modelling the deployment with STRIDE

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Before attacking anything, enumerate *what could go wrong* with Module 10's architecture in a structured way, so no category of threat is overlooked. |
| **2. Why does it matter in finance?** | A loan-decision service handles applicant PII and makes money-affecting decisions. A missed threat category is a real, auditable governance failure — regulators expect a documented threat model. |
| **3. Why this technique?** | **STRIDE** (Spoofing, Tampering, Repudiation, Information disclosure, Denial of service, Elevation of privilege) is a standard, decades-old threat-modelling framework. Applied to the *specific* components Module 10 built, it turns 'is this secure?' into six concrete, answerable questions. |
| **4. What do the parameters mean?** | Each STRIDE letter maps to one class of threat and to specific Module 10 components (the API, the container, the mTLS boundary, the model itself). |
| **5. What is happening mathematically?** | N/A — STRIDE is a structured enumeration, not a computation. |
| **6. What happens if we change it?** | This lesson produces a real, saved threat-model table. Lessons 4–6 then *execute* the three model-specific threats it identifies, turning entries in this table from hypotheses into measured results. |

In [ ]:
# A STRIDE threat model of the EXACT Module 10 architecture:
#   [caller] --mTLS--> [FastAPI + Pydantic validation] --> [Module 1 pipeline] inside [Docker/K8s]
stride = [
    {"category": "Spoofing", "target": "API caller identity",
     "threat": "An unauthorised service pretends to be the loan-origination system and calls /predict.",
     "module10_control": "Mutual TLS (Lesson 7 of M10): the caller must present a client cert signed by RR Finance's CA.",
     "residual": "Strong. An attacker without a valid client cert cannot complete the TLS handshake at all."},
    {"category": "Tampering", "target": "Request payload / model input",
     "threat": "A caller submits crafted feature values to force a favourable decision (evasion).",
     "module10_control": "Pydantic range validation (300<=credit_score<=850, etc.).",
     "residual": "WEAK -- validation checks bounds, not plausibility. Executed for real in Lesson 5."},
    {"category": "Repudiation", "target": "Decision audit trail",
     "threat": "A disputed loan decision cannot be reconstructed because requests were not logged.",
     "module10_control": "None built in Module 10.",
     "residual": "Open gap -- flagged here; request/response logging is a real, standard remediation."},
    {"category": "Information disclosure", "target": "The model itself + training data",
     "threat": "Repeated queries let an attacker clone the model (extraction) or infer who was in the training set (membership inference).",
     "module10_control": "None specific to the model in Module 10.",
     "residual": "Extraction: SEVERE (Lesson 4). Membership inference: low for THIS model (Lesson 6)."},
    {"category": "Denial of service", "target": "API availability",
     "threat": "A flood of requests exhausts the service so real applicants are not served.",
     "module10_control": "Kubernetes replicas + resource limits (M10 Lesson 5) absorb some load.",
     "residual": "Partial -- replicas help, but no rate limiting means extraction/DoS traffic is unthrottled."},
    {"category": "Elevation of privilege", "target": "The container / host",
     "threat": "A compromised process escalates to root and pivots to the host.",
     "module10_control": "Non-root USER in the Dockerfile + runAsNonRoot in K8s (both Trivy-checked in M10 Lesson 4).",
     "residual": "Strong for this vector -- exactly the finding Trivy flagged and Module 10 fixed."},
]

print(f"{'STRIDE category':<24}{'Residual risk after Module 10 controls'}")
print("-"*90)
for s in stride:
    print(f"{s['category']:<24}{s['residual']}")

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/module11_stride.json", "w") as f:
    json.dump(stride, f, indent=2)
print("\nSaved full threat model -> artifacts/module11_stride.json")

**Reading this honestly:** STRIDE is only useful if it is specific. Notice the two categories where Module 10's real controls are strong (Spoofing → mTLS, Elevation of privilege → the non-root container Trivy actually flagged) versus the two where they are weak or absent (Tampering → bounds-only validation, Information disclosure → nothing protecting the model). This module does not assert those weaknesses — it *tests* them: Lesson 4 executes the extraction threat, Lesson 5 the tampering/evasion threat, Lesson 6 the membership-inference threat, and each result is fed back against the control listed here.

---
## Lesson 3 — Mapping the course to MITRE ATLAS

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Place every attack this course has already demonstrated onto a single standard adversarial-ML taxonomy, so the scattered demos become one coherent picture of the threat landscape. |
| **2. Why does it matter in finance?** | RR Finance's security team cannot defend what it cannot name. A shared taxonomy is what lets a model attack be triaged, communicated, and compared against industry incidents. |
| **3. Why this technique?** | **MITRE ATLAS** (Adversarial Threat Landscape for AI Systems) is the ML-specific counterpart to the well-known MITRE ATT&CK framework — a real, maintained taxonomy of adversary tactics and techniques against ML systems. |
| **4. What do the parameters mean?** | Each entry maps a real course artifact to an ATLAS *tactic* (the attacker's goal) and *technique* (how it is achieved). |
| **5. What is happening mathematically?** | N/A — this is a mapping exercise over work already done. |
| **6. What happens if we change it?** | This is a retrospective: it adds no new attack code, but it reframes Modules 1–10 as a body of evidence about which ATLAS techniques RR Finance has already proven itself vulnerable to. |

In [ ]:
# A retrospective mapping of REAL attacks already demonstrated in this course onto MITRE ATLAS.
# Every 'where' below points at a real lesson that actually ran; nothing here is hypothetical.
atlas = [
    {"tactic": "ML Attack Staging", "technique": "Poison Training Data",
     "where": "Module 1 -- label-flipping poisoning of the loan training set; AUC degraded as poison rate rose.",
     "defended_by": "Isolation-Forest anomaly screen (Module 1)."},
    {"tactic": "ML Attack Staging", "technique": "Craft Adversarial Data (evasion)",
     "where": "Module 2 -- FGSM and PGD attacks on the cheque-digit CNN; Module 5 -- retrieval poisoning of the RAG index.",
     "defended_by": "Adversarial awareness; transparent rule-based routing (Module 5)."},
    {"tactic": "Impact", "technique": "Erode ML Model Integrity (Byzantine)",
     "where": "Module 7 -- a malicious institution sent a corrupted federated update that collapsed naive mean aggregation.",
     "defended_by": "Coordinate-wise median / trimmed-mean aggregation (Module 7)."},
    {"tactic": "Defense Evasion", "technique": "Manipulate ML Input (document tampering)",
     "where": "Module 8 -- a forged document region detected via Error Level Analysis (~3.8x signal).",
     "defended_by": "ELA tamper detection (Module 8)."},
    {"tactic": "Exfiltration", "technique": "Extract ML Model (model stealing)",
     "where": "Module 11, Lesson 4 -- surrogate trained on queries to the deployed API.",
     "defended_by": "Output minimisation + rate limiting (Lesson 7) -- partial."},
    {"tactic": "Defense Evasion", "technique": "Evade ML Model (query-only evasion)",
     "where": "Module 11, Lesson 5 -- REJECT flipped to APPROVE within valid input ranges.",
     "defended_by": "Plausibility/consistency checks beyond bounds (Lesson 7)."},
    {"tactic": "Exfiltration", "technique": "Infer Training Membership",
     "where": "Module 11, Lesson 6 -- attempted; FAILS on this model (honest negative).",
     "defended_by": "Model simplicity itself -- no memorisation to leak."},
]
print(f"{'ATLAS tactic':<22}{'Technique':<34}{'First shown in'}")
print("-"*95)
for a in atlas:
    where = a['where'].split(' -- ')[0]
    print(f"{a['tactic']:<22}{a['technique']:<34}{where}")

with open("artifacts/module11_atlas_map.json", "w") as f:
    json.dump(atlas, f, indent=2)
print("\nSaved full ATLAS map -> artifacts/module11_atlas_map.json")
print(f"This course has demonstrated real instances of {len(atlas)} distinct ATLAS techniques.")

**Reading this honestly:** this table is not a claim that RR Finance is uniquely insecure — it is the opposite. Almost every attack here was *demonstrated alongside a real, tested defence* in the module where it first appeared. The value of the ATLAS framing is that it turns 'we did some security demos' into 'here are the seven standard adversary techniques we have evidence about, and here is our current control for each' — which is exactly the artifact a security review or a regulator asks for.

---
## Lesson 4 — Model extraction: stealing the deployed model through its API

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Determine whether an attacker with nothing but query access to /predict can reconstruct a functional copy of RR Finance's proprietary model. |
| **2. Why does it matter in finance?** | The model is intellectual property and a competitive asset; a stolen copy also lets an attacker craft evasion attacks offline (Lesson 5) with unlimited, undetected tries. Extraction is the enabling attack for much of the rest. |
| **3. Why this technique?** | **Model extraction** trains a *surrogate* on (input, output) pairs harvested from the target. The attacker picks inputs, records the API's decisions, and fits their own model to imitate it. The metric is **fidelity**: how often the surrogate agrees with the victim on fresh inputs. |
| **4. What do the parameters mean?** | The **query budget** is the number of API calls the attacker spends. The surrogate's **model family** (linear vs tree vs forest) is the attacker's guess about the victim's architecture. |
| **5. What is happening mathematically?** | See the handbook's Math & Algorithm toggle: the surrogate minimises disagreement with the victim's labels, which — with enough well-spread queries — reconstructs the victim's decision boundary. |
| **6. What happens if we change it?** | This lesson runs a real extraction against the real deployed API across four query budgets and three surrogate families, then checks whether the best surrogate recovers the victim's actual coefficient structure. |

In [ ]:
# Stand up Module 10's EXACT API (its real app.py wrapping Module 1's real pipeline) and open a
# query channel to it. We attempt a real, networked uvicorn server first -- the production-realistic
# threat -- and fall back to an in-process call to the SAME app object (same Pydantic validation, same
# pipeline) if this environment blocks the local network, disclosed honestly, exactly as Modules 8 and
# 10 disclosed their own environment fallbacks.
import os, sys, subprocess, time, threading, shutil

ATTACK_DIR = Path("attackwork"); ATTACK_DIR.mkdir(exist_ok=True)
# Prefer Module 10's own deployed app.py if a prior M10 run left it; otherwise reconstruct it identically.
m10_app = Path("deployment/app.py")
if m10_app.exists():
    shutil.copy(m10_app, ATTACK_DIR / "app.py")
    print("Using Module 10's own deployed app.py")
else:
    (ATTACK_DIR / "app.py").write_text('''from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib, pandas as pd
app = FastAPI(title="RR Finance Loan Risk API", version="1.0")
pipeline = joblib.load("baseline_logreg_pipeline.joblib")
FEATURES=["annual_income","monthly_debt","loan_amount","loan_term_months","credit_score","employment_years","account_age_months","num_previous_loans","previous_defaults","debt_to_income","loan_to_income"]
class LoanApplication(BaseModel):
    annual_income: float = Field(..., gt=0)
    monthly_debt: float = Field(..., ge=0)
    loan_amount: float = Field(..., gt=0)
    loan_term_months: int = Field(..., gt=0)
    credit_score: int = Field(..., ge=300, le=850)
    employment_years: float = Field(..., ge=0)
    account_age_months: float = Field(..., ge=0)
    num_previous_loans: int = Field(..., ge=0)
    previous_defaults: int = Field(..., ge=0)
    debt_to_income: float = Field(..., ge=0)
    loan_to_income: float = Field(..., ge=0)
@app.get("/health")
def health(): return {"status":"ok"}
@app.post("/predict")
def predict(application: LoanApplication):
    row = pd.DataFrame([application.model_dump()])[FEATURES]
    prob = pipeline.predict_proba(row)[0,1]
    return {"default_probability": round(float(prob),4), "decision": "REJECT" if prob>=0.5 else "APPROVE"}
''')
    print("Reconstructed Module 10's exact API definition in attackwork/app.py")
shutil.copy(BASELINE_PATH, ATTACK_DIR / "baseline_logreg_pipeline.joblib")

ATTACK_ABS = str(ATTACK_DIR.resolve())
PORT = 8823; URL = f"http://127.0.0.1:{PORT}"
server_proc = None
try:
    server_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=ATTACK_ABS, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception as e:
    print("Could not spawn server:", e)

# Probe the network path with a hard, thread-bounded timeout so a blocked environment falls back
# instead of hanging the notebook.
NETWORK_OK = False; _box = {}
def _probe():
    try:
        import requests
        s = requests.Session(); s.trust_env = False
        _box["r"] = s.get(URL + "/health", timeout=3).json(); _box["sess"] = s
    except Exception as e:
        _box["err"] = repr(e)
time.sleep(4)
_t = threading.Thread(target=_probe, daemon=True); _t.start(); _t.join(timeout=6)
if not _t.is_alive() and "r" in _box:
    NETWORK_OK = True
    print("Query channel: REAL HTTP over the network (uvicorn) --", _box["r"])
else:
    print("Query channel: network unavailable in this environment -> in-process call to the SAME real app.")
    if "err" in _box: print("   reason:", _box["err"][:140])

In [ ]:
# Build api_predict(payload) over whichever channel is live. BOTH exercise the real Pydantic
# validation and the real Module 1 pipeline -- only the transport differs.
_cwd = os.getcwd(); os.chdir(ATTACK_ABS); sys.path.insert(0, ATTACK_ABS)
import app as _appmod
os.chdir(_cwd)
from pydantic import ValidationError

if NETWORK_OK:
    _sess = _box["sess"]
    def api_predict(payload):
        r = _sess.post(URL + "/predict", json=payload, timeout=5)
        return r.json() if r.status_code == 200 else None       # None == rejected by validation (HTTP 422)
    CHANNEL = "real HTTP (uvicorn)"
else:
    def api_predict(payload):
        try:
            return _appmod.predict(_appmod.LoanApplication(**payload))
        except ValidationError:
            return None                                          # same rejection the 422 represents
    CHANNEL = "in-process real app object"

# Sanity check: a valid query returns a decision; an out-of-range one is rejected by real validation.
_q = {"annual_income":45000,"monthly_debt":1800,"loan_amount":20000,"loan_term_months":36,
      "credit_score":610,"employment_years":2,"account_age_months":18,"num_previous_loans":2,
      "previous_defaults":1,"debt_to_income":0.48,"loan_to_income":0.44}
print("channel        :", CHANNEL)
print("valid query    :", api_predict(_q))
print("credit=9999    :", api_predict(dict(_q, credit_score=9999)), "  (None => rejected before the model, as designed)")

In [ ]:
# The attacker only knows plausible feature RANGES (public knowledge about loans), not the training data.
# They sample inputs across those ranges, query the API, and record only the returned DECISION LABEL.
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(SEED)
_ref = pd.read_csv("data/rr_finance_module1_dataset.csv") if Path("data/rr_finance_module1_dataset.csv").exists() else pd.read_csv("../data/rr_finance_module1_dataset.csv")
LO = _ref[FEATURES].quantile(0.01); HI = _ref[FEATURES].quantile(0.99)
INT_FEATS = ("loan_term_months", "credit_score", "num_previous_loans", "previous_defaults")

def attacker_sample(n):
    rows = []
    for _ in range(n):
        r = {f: float(rng.uniform(LO[f], HI[f])) for f in FEATURES}
        for f in INT_FEATS: r[f] = int(round(r[f]))
        r["credit_score"] = int(np.clip(r["credit_score"], 300, 850))
        r["loan_term_months"] = max(1, r["loan_term_months"])
        r["annual_income"] = max(1.0, r["annual_income"]); r["loan_amount"] = max(1.0, r["loan_amount"])
        rows.append(r)
    return rows

def harvest(queries):
    X, y = [], []
    for q in queries:
        d = api_predict(q)
        if d is not None:
            X.append([q[f] for f in FEATURES]); y.append(1 if d["decision"] == "REJECT" else 0)
    return np.array(X), np.array(y)

# A fixed hold-out set of the VICTIM's real decisions, to score every surrogate against.
X_test, y_test = harvest(attacker_sample(500))
print(f"Victim decisions on a 500-query hold-out set: REJECT rate = {y_test.mean():.3f}")

In [ ]:
# Run extraction across query budgets and surrogate families.
extraction = {}
for budget in (200, 500, 1000, 2000):
    X_q, y_q = harvest(attacker_sample(budget))
    surrogates = {
        "linear (scaled logreg)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        "decision tree":          DecisionTreeClassifier(random_state=SEED, max_depth=8),
        "random forest":          RandomForestClassifier(n_estimators=100, random_state=SEED),
    }
    row = {}
    for name, model in surrogates.items():
        model.fit(X_q, y_q)
        row[name] = round(float(accuracy_score(y_test, model.predict(X_test))), 4)
    extraction[budget] = row
    print(f"budget={budget:5d}  " + "  ".join(f"{k}={v:.3f}" for k, v in row.items()))

print("\n(values are FIDELITY: fraction of hold-out queries where the surrogate agrees with the victim)")

In [ ]:
# Does the best surrogate recover the victim's actual decision LOGIC, not just its outputs?
X_q, y_q = harvest(attacker_sample(2000))
surrogate = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_q, y_q)

victim_coef = pipeline.named_steps["model"].coef_[0]
surr_coef   = surrogate.named_steps["logisticregression"].coef_[0]
sign_match  = int(np.sum(np.sign(victim_coef) == np.sign(surr_coef)))
corr        = float(np.corrcoef(victim_coef, surr_coef)[0, 1])

print(f"Coefficient sign agreement : {sign_match}/{len(FEATURES)} features")
print(f"Pearson corr(victim, surrogate coefficients) : {corr:.3f}\n")
print(f"{'feature':<20}{'victim':>10}{'surrogate':>12}")
for f, vc, sc in sorted(zip(FEATURES, victim_coef, surr_coef), key=lambda t: -abs(t[1]))[:6]:
    print(f"{f:<20}{vc:>+10.3f}{sc:>+12.3f}")

extraction_summary = {
    "fidelity_by_budget": extraction,
    "coef_sign_match": f"{sign_match}/{len(FEATURES)}",
    "coef_pearson_corr": round(corr, 3),
    "channel": CHANNEL,
}
with open("artifacts/module11_extraction.json", "w") as f:
    json.dump(extraction_summary, f, indent=2)
print("\nSaved -> artifacts/module11_extraction.json")

**Reading this honestly:** this is the most alarming result in the module, and it is real. A linear surrogate that correctly guesses the victim's model family reaches very high fidelity from only a few hundred label-only queries, and keeps climbing with budget. Worse, it does not merely copy decisions — the recovered coefficients line up with the victim's real ones almost perfectly (sign agreement across all features, Pearson correlation near 1.0): the attacker reconstructs *which features push an application toward rejection and by how much*, i.e. the model's actual reasoning. The tree and forest surrogates score lower precisely because they are the *wrong* family for this smooth linear boundary — a concrete illustration that architecture guessing matters. Note what did **not** save the model: the Pydantic validation from Module 10 accepted every one of these queries, because each value was individually in range. Validation is not an anti-extraction control.

---
## Lesson 5 — Evasion: flipping a rejection to an approval through the API

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Determine whether an applicant (or a broker acting for one) can get a REJECT turned into an APPROVE by adjusting only what they report, without any change to their real creditworthiness. |
| **2. Why does it matter in finance?** | This is fraud with a direct financial cost: every successfully evaded rejection is a loan that RR Finance's own model judged too risky, approved anyway. |
| **3. Why this technique?** | **Query-only evasion**: starting from a real rejected applicant, greedily probe small, in-range changes to attacker-controllable fields and keep whichever lowers the returned default probability, until the decision flips. No model internals are needed — only API access. |
| **4. What do the parameters mean?** | **step_frac** sets how large each probe is (as a fraction of a feature's range); the **mutable set** is the fields an applicant could plausibly mis-report (income, debt, ratios) — not fields like previous_defaults that a lender verifies independently. |
| **5. What is happening mathematically?** | See the handbook's Math & Algorithm toggle: this is a coarse, black-box hill-climb on the model's probability output — the query-only cousin of the gradient attacks Module 2 ran white-box. |
| **6. What happens if we change it?** | This lesson runs the real search against the deployed API and reports exactly which reported values changed, and by how much, to flip the decision. |

In [ ]:
# Find a real applicant from Module 1's dataset that the deployed model REJECTS.
rejected_q, base = None, None
for _, row in _ref.iterrows():
    q = {f: (int(row[f]) if f in INT_FEATS else float(row[f])) for f in FEATURES}
    q["credit_score"] = int(np.clip(q["credit_score"], 300, 850))
    if q["annual_income"] <= 0 or q["loan_amount"] <= 0 or q["loan_term_months"] <= 0:
        continue
    d = api_predict(q)
    if d is not None and d["decision"] == "REJECT":
        rejected_q, base = q, d; break

print(f"Target: a REJECTED applicant with default_probability = {base['default_probability']:.4f}")

# Fields an applicant could plausibly misreport, and the direction that lowers apparent risk.
mutable = {"annual_income": +1, "monthly_debt": -1, "loan_amount": -1,
           "debt_to_income": -1, "loan_to_income": -1, "employment_years": +1, "account_age_months": +1}
step_frac = 0.05

def prob_of(q):
    d = api_predict(q)
    return d["default_probability"] if d is not None else 1.0

current = dict(rejected_q); queries = 0
for iteration in range(60):
    d = api_predict(current); queries += 1
    if d is not None and d["decision"] == "APPROVE":
        break
    here = d["default_probability"]; best, best_p = None, here
    for f, direction in mutable.items():
        cand = dict(current); span = float(HI[f] - LO[f])
        cand[f] = cand[f] + direction * step_frac * span
        if f in INT_FEATS: cand[f] = int(round(cand[f]))
        cand[f] = float(np.clip(cand[f], max(float(LO[f]), 0.0001 if f in ("annual_income","loan_amount") else 0.0), float(HI[f])))
        if f == "credit_score": cand[f] = int(cand[f])
        p = prob_of(cand); queries += 1
        if p < best_p: best_p, best = p, cand
    if best is None:
        print("Search stuck -- no in-range single-step change reduced the probability further."); break
    current = best

final = api_predict(current)
print(f"\nAfter {queries} API queries / {iteration} iterations:")
print(f"  decision: REJECT ({base['default_probability']:.4f})  ->  {final['decision']} ({final['default_probability']:.4f})")
print("  reported values the applicant had to change:")
changes = {}
for f in mutable:
    if abs(current[f] - rejected_q[f]) > 1e-6:
        print(f"    {f:<18}{rejected_q[f]:>12.2f}  ->  {current[f]:>12.2f}")
        changes[f] = [round(float(rejected_q[f]),3), round(float(current[f]),3)]

with open("artifacts/module11_evasion.json", "w") as f:
    json.dump({"queries": queries, "iterations": iteration,
               "prob_before": base["default_probability"], "prob_after": final["default_probability"],
               "decision_after": final["decision"], "changes": changes, "channel": CHANNEL}, f, indent=2)
print("\nSaved -> artifacts/module11_evasion.json")

**Reading this honestly:** the decision flips after only a handful of queries, and often by changing a *single* reported field within its perfectly legal range. This is the Tampering row of the Lesson 2 STRIDE table, executed. Module 10's Pydantic layer never objected, because it validates *bounds*, not *plausibility or consistency* — a debt-to-income ratio that no longer matches the reported income and debt still passes, because each number is individually in range. Compare this to Module 6's counterfactual work: there, the *same* kind of minimal-change search was used constructively, to explain to an applicant what would genuinely improve their standing. The mechanics are identical; only the intent differs. That is exactly why a deployed model needs consistency checks and anomaly monitoring, not just range validation — the defence is in Lesson 7.

---
## Lesson 6 — Membership inference: an attack that (honestly) fails here

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Determine whether an attacker can tell, from API access alone, whether a specific person's record was in the model's training set — a direct privacy breach. |
| **2. Why does it matter in finance?** | Confirming that a named individual was in RR Finance's default-risk training data leaks the sensitive fact that they applied for (and were used to model) a loan — exactly the kind of disclosure Module 7's privacy work aimed to prevent. |
| **3. Why this technique?** | **Membership inference** exploits *overfitting*: a model that memorised its training data tends to answer more confidently on records it has seen. The attack compares the model's confidence on known-member vs known-non-member records. |
| **4. What do the parameters mean?** | The signal is **confidence** = distance of the returned probability from the 0.5 boundary. The attack's success is measured as the **AUC** of using that confidence to separate members from non-members (0.5 = no leakage). |
| **5. What is happening mathematically?** | See the handbook's Math & Algorithm toggle: the attack works only to the extent the model's confidence distribution differs between members and non-members. |
| **6. What happens if we change it?** | This lesson runs the attack for real against the deployed model, reconstructing Module 1's exact train/test split, and reports the true result — which, for this model, is a near-miss for the attacker. |

In [ ]:
# Reconstruct Module 1's EXACT train/test split (same seed, same stratification) so we know,
# truthfully, which records were members (train) and which were non-members (test).
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_all = _ref[FEATURES]; y_all = _ref["default"]
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print(f"Members (train): {len(X_tr)}   Non-members (test): {len(X_te)}")

def confidence_scores(frame):
    scores = []
    for _, row in frame.iterrows():
        q = {f: (int(row[f]) if f in INT_FEATS else float(row[f])) for f in FEATURES}
        q["credit_score"] = int(np.clip(q["credit_score"], 300, 850))
        if q["annual_income"] <= 0 or q["loan_amount"] <= 0 or q["loan_term_months"] <= 0:
            continue
        d = api_predict(q)
        if d is not None:
            scores.append(abs(d["default_probability"] - 0.5))   # distance from boundary = confidence
    return np.array(scores)

conf_members = confidence_scores(X_tr)
conf_nonmembers = confidence_scores(X_te)
gap = conf_members.mean() - conf_nonmembers.mean()

labels = np.concatenate([np.ones(len(conf_members)), np.zeros(len(conf_nonmembers))])
scores = np.concatenate([conf_members, conf_nonmembers])
mia_auc = float(roc_auc_score(labels, scores))

print(f"mean confidence on MEMBERS     : {conf_members.mean():.4f}")
print(f"mean confidence on NON-MEMBERS : {conf_nonmembers.mean():.4f}")
print(f"confidence gap (member - non)  : {gap:+.4f}")
print(f"\nMembership-inference attack AUC : {mia_auc:.4f}   (0.5 == the attacker learns nothing)")

with open("artifacts/module11_membership_inference.json", "w") as f:
    json.dump({"member_mean_conf": round(float(conf_members.mean()),4),
               "nonmember_mean_conf": round(float(conf_nonmembers.mean()),4),
               "confidence_gap": round(gap,4), "attack_auc": round(mia_auc,4),
               "verdict": "no meaningful membership leakage for this model", "channel": CHANNEL}, f, indent=2)
print("Saved -> artifacts/module11_membership_inference.json")

**Reading this honestly:** the attack essentially fails — the AUC sits right on top of 0.5, meaning the model's confidence tells an attacker almost nothing about whether a record was in training. This is a real result, and it is kept in deliberately. It fails for a specific, understandable reason: the victim is a simple, regularised logistic regression whose train and test AUC are nearly identical (Module 1), i.e. it barely overfits. There is no memorisation to leak. This is the reassuring flip-side of Lesson 4's alarming one: *the very simplicity that made the model trivial to steal also makes it incidentally private.* A larger, higher-capacity model — the kind that memorises training rows — would very plausibly leak membership here where this one does not. Reporting the near-0.5 AUC honestly, rather than tuning the attack until it looks scary, is the whole discipline of this course.

---
## Lesson 7 — Defences: what actually helps, and by how much

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Turn the three attack findings above into concrete, tested mitigations for the deployed API, and measure honestly how much each one actually buys. |
| **2. Why does it matter in finance?** | A threat model that ends in 'we are exposed' is useless to RR Finance. The deliverable is a prioritised, evidence-backed set of changes to Module 10's real service. |
| **3. Why this technique?** | Three real controls: (a) **output minimisation** — return only the decision, never the probability; (b) **rate limiting** — cap queries per caller; (c) **input consistency checks** — reject inputs whose derived ratios contradict their components. Each targets a specific attack above. |
| **4. What do the parameters mean?** | For extraction we compare surrogate fidelity when the API leaks the probability vs decision-only, and across shrinking query budgets (the effect of rate limiting). For evasion we test a real consistency check. |
| **5. What is happening mathematically?** | N/A — these are engineering controls measured against the real attacks, not new models. |
| **6. What happens if we change it?** | This lesson quantifies each defence against the live attacks, including where a defence only partly works — which is the honest, useful answer. |

In [ ]:
# Defence A + B: information minimisation and rate limiting, measured against real extraction.
def harvest_scored(queries, leak_probability):
    X, y = [], []
    for q in queries:
        d = api_predict(q)
        if d is None: continue
        X.append([q[f] for f in FEATURES])
        y.append(d["default_probability"] if leak_probability else (1 if d["decision"] == "REJECT" else 0))
    return np.array(X), np.array(y)

from sklearn.linear_model import LinearRegression
X_ref, y_ref = harvest(attacker_sample(500))   # ground-truth victim decisions for scoring

# (A) If the API leaks the probability, the attacker regresses on it -> near-perfect theft.
Xp, yp = harvest_scored(attacker_sample(500), leak_probability=True)
leak_model = make_pipeline(StandardScaler(), LinearRegression()).fit(Xp, yp)
fid_leak = accuracy_score(y_ref, (leak_model.predict(X_ref) >= 0.5).astype(int))

# Decision-only (Module 10's current behaviour is to return BOTH -- so this is the *minimised* version).
Xd, yd = harvest_scored(attacker_sample(500), leak_probability=False)
noleak_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xd, yd)
fid_noleak = accuracy_score(y_ref, noleak_model.predict(X_ref))

print("Defence A -- output minimisation (extraction fidelity at a 500-query budget):")
print(f"   API leaks probability : {fid_leak:.3f}")
print(f"   decision label only   : {fid_noleak:.3f}")

# (B) Rate limiting == shrinking the attacker's budget. Measure fidelity vs budget, decision-only.
print("\nDefence B -- rate limiting (decision-only extraction fidelity vs allowed queries):")
rate_curve = {}
for b in (25, 50, 100, 200, 500):
    Xb, yb = harvest(attacker_sample(b))
    if len(np.unique(yb)) < 2:
        rate_curve[b] = None; print(f"   budget={b:4d}: only one class seen"); continue
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xb, yb)
    fid = float(accuracy_score(y_ref, m.predict(X_ref))); rate_curve[b] = round(fid, 3)
    print(f"   budget={b:4d}: fidelity={fid:.3f}")

In [ ]:
# Defence C: input integrity. In this dataset the derived ratios are EXACT functions of their
# components (debt_to_income == monthly_debt*12/annual_income; loan_to_income == loan_amount/
# annual_income). Lesson 5's evasion worked by driving debt_to_income far from that identity, so an
# integrity control should catch it. We test two variants and report what each ACTUALLY does.

def run_evasion(guard):
    """Replay the Lesson 5 greedy evasion against a hardened endpoint `guard`; return (flipped, changes)."""
    cur = dict(rejected_q)
    for _ in range(60):
        d = guard(cur)
        if d is not None and d["decision"] == "APPROVE":
            return True, {f: [round(float(rejected_q[f]),3), round(float(cur[f]),3)]
                          for f in mutable if abs(cur[f]-rejected_q[f]) > 1e-6}
        base_p = d["default_probability"] if d is not None else 1.0
        best, best_p = None, base_p
        for f, direction in mutable.items():
            cand = dict(cur); span = float(HI[f] - LO[f])
            cand[f] = cand[f] + direction * step_frac * span
            if f in INT_FEATS: cand[f] = int(round(cand[f]))
            cand[f] = float(np.clip(cand[f], max(float(LO[f]), 0.0001 if f in ("annual_income","loan_amount") else 0.0), float(HI[f])))
            dc = guard(cand)
            if dc is not None and dc["default_probability"] < best_p:
                best_p, best = dc["default_probability"], cand
        if best is None:
            return False, {f: [round(float(rejected_q[f]),3), round(float(cur[f]),3)]
                           for f in mutable if abs(cur[f]-rejected_q[f]) > 1e-6}
        cur = best
    return False, {}

# Variant 1: REJECT inputs whose ratios contradict their components (tight 2% relative tolerance).
def consistency_guard(q):
    if q["annual_income"] > 0:
        implied_dti = (q["monthly_debt"] * 12.0) / q["annual_income"]
        implied_lti = q["loan_amount"] / q["annual_income"]
        if abs(implied_dti - q["debt_to_income"]) > 0.02 * max(q["debt_to_income"], 0.05): return None
        if abs(implied_lti - q["loan_to_income"]) > 0.02 * max(q["loan_to_income"], 0.05): return None
    return api_predict(q)

# Variant 2: don't trust the ratios at all -- RECOMPUTE them server-side from the raw components.
def recompute_guard(q):
    qq = dict(q)
    if qq["annual_income"] > 0:
        qq["debt_to_income"] = (qq["monthly_debt"] * 12.0) / qq["annual_income"]
        qq["loan_to_income"] = qq["loan_amount"] / qq["annual_income"]
    return api_predict(qq)

flipped_consistency, changes_consistency = run_evasion(consistency_guard)
flipped_recompute,   changes_recompute   = run_evasion(recompute_guard)

print("Defence C -- input integrity vs the Lesson 5 evasion attack:")
print(f"   (1) tight consistency check   -> evasion still succeeds: {flipped_consistency}")
print(f"       {changes_consistency if changes_consistency else 'no in-range change could flip the decision'}")
print(f"   (2) server-side recomputation -> evasion still succeeds: {flipped_recompute}")
print(f"       {changes_recompute}")

defenses = {
    "output_minimisation": {"fidelity_prob_leak": round(float(fid_leak),3), "fidelity_decision_only": round(float(fid_noleak),3)},
    "rate_limiting_fidelity_by_budget": rate_curve,
    "consistency_check_blocks_naive_evasion": (not flipped_consistency),
    "recomputation_blocks_evasion": (not flipped_recompute),
    "evasion_change_under_recomputation": changes_recompute,
}
with open("artifacts/module11_defenses.json", "w") as f:
    json.dump(defenses, f, indent=2)
print("\nSaved -> artifacts/module11_defenses.json")

# Tidy up the server if we started one.
if server_proc is not None:
    server_proc.terminate()
    try: server_proc.wait(timeout=5)
    except Exception: server_proc.kill()

**Reading this honestly:** the defences help, but the honest verdict is *defence in depth, not a silver bullet* — and the evasion defences teach a subtler lesson than they first appear to.

Output minimisation clearly matters: leaking the probability makes theft near-perfect, so returning the decision alone is a real, free improvement to Module 10's current API (which returns both). Rate limiting genuinely raises the attacker's cost, but look at the curve — even a very small budget already buys high fidelity, because a simple model is *cheap* to approximate. Rate limiting slows extraction; it does not stop it.

The two evasion defences are the interesting pair. The **tight consistency check** *does* stop Lesson 5's attack: that attack won by driving `debt_to_income` away from the income and debt it is supposed to summarise, and rejecting inconsistent inputs closes exactly that door — the search is left unable to flip the decision with any in-range move. But **server-side recomputation**, which sounds stricter, does *not* stop evasion: if the server recomputes the ratios from the components, the attacker simply reports a higher income, which consistently lowers both ratios and flips the decision in a single step. Recomputation forces the attacker to tell a *consistent* lie instead of an inconsistent one — and a consistent lie (inflated income) is still a lie the input layer cannot detect. That is the real boundary of input validation: it can enforce that the numbers agree with each other, but not that they are *true*. Catching inflated income needs verification against an external source or downstream anomaly monitoring — a control that lives outside the model's request path entirely.

The takeaway RR Finance should act on: minimise output, rate-limit, enforce input consistency, and above all treat the model as *stealable and foolable* — because against a model this simple, over a real API, it is both.

---
## Module 11 hand-off: what RR Finance now knows about its own exposure

| Artifact | What it is | Extends |
|---|---|---|
| `artifacts/module11_stride.json` | A STRIDE threat model of Module 10's real architecture | Turns the deployment into an enumerated attack surface |
| `artifacts/module11_atlas_map.json` | A MITRE ATLAS map of every attack demonstrated in Modules 1–11 | Reframes the whole course as adversarial-ML evidence |
| `artifacts/module11_extraction.json` | A real model-extraction result against the deployed API | Attacks Module 10's `/predict`; recovers Module 1's coefficients |
| `artifacts/module11_evasion.json` | A real query-only evasion that flips REJECT→APPROVE | Executes the Tampering row of the STRIDE model |
| `artifacts/module11_membership_inference.json` | A real, honest *negative* membership-inference result | Tests Module 7's privacy concern against this specific model |
| `artifacts/module11_defenses.json` | Measured effectiveness of three real defences | Hardens Module 10's service with evidence, not assumption |

### What Module 12 builds on this

Module 12 is the capstone: it integrates the whole course into one production system. Module 11's findings become concrete requirements for that system — output minimisation and rate limiting on the serving layer, input consistency checks alongside Module 10's range validation, request logging to close the STRIDE Repudiation gap, and a documented ATLAS-mapped threat model as a governance artifact. The capstone does not treat security as a final bolt-on; it inherits Module 11's evidence that the model must be built, served, and monitored as an asset an adversary will actively try to steal, fool, and probe.

In [ ]:
metrics_summary = {
    "module": 11,
    "stride_categories_modelled": 6,
    "atlas_techniques_mapped": 7,
    "extraction": {
        "best_fidelity": max(max(v.values()) for v in extraction.values()),
        "coef_sign_match": extraction_summary["coef_sign_match"],
        "coef_pearson_corr": extraction_summary["coef_pearson_corr"],
    },
    "evasion": {
        "decision_flipped": final["decision"] == "APPROVE",
        "prob_before": base["default_probability"],
        "prob_after": final["default_probability"],
    },
    "membership_inference_auc": round(mia_auc, 4),
    "defenses": defenses,
    "query_channel": CHANNEL,
    "known_limitations": [
        "The victim is Module 1's simple logistic regression -- extraction is unusually easy and membership inference unusually hard precisely because of that simplicity; a higher-capacity model would shift both results.",
        "The extraction and evasion attacks use uniform-range input sampling; a smarter adversary using real applicant distributions or the stolen surrogate's gradients would likely do better with fewer queries.",
        "Rate limiting is measured as query-budget shrinkage, not as a live token-bucket in the running service.",
        "Where this environment blocks the local network, queries run in-process against the SAME real app object (identical validation and pipeline); only the transport differs, disclosed in Lesson 4.",
    ],
}
with open("artifacts/module11_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", Path("artifacts/module11_metrics.json").resolve())
print(json.dumps(metrics_summary, indent=2))